```
Homework Assignment #5
Mahima Kumar
120874512
CMSC320 0101
```

In [21]:
# imports! referencing piazza post @195
!pip install -q requests pandas matplotlib scikit-learn transformers torch torchvision
import requests
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, mean_absolute_error
from transformers import pipeline

In [22]:
# these r the profs im choosing for this proj!
profs = [
  "Nora Burkhauser", # 132 javaaa
  "Larry Herman", # 216 ok linux fr
  "Maksym Morawski", # my goat for 250 n 320
  "Clyde Kruskal", # 351 was an era
  "Nelson Padua-Perez" # 335 clutch
]

In [23]:
# to reference planet terp im using this api link
APIURL = "https://planetterp.com/api/v1/"

In [24]:
# this function gets the reviews for each prof
def getReviews(name):

  # this looks for the profs name and if reviews
  url = APIURL + "professor"
  params = {
    "name": name,
    "reviews": "true"
  }

  # checks if the url worked
  response = requests.get(url, params = params)

  # if the url times out then it cant get
  if response.status_code != 200:
    print("Cant get", name, response.status_code)
    return []

  # else if no reviews, then says no reviews
  data = response.json()
  if "reviews" not in data:
    print("No reviews for:", name)
    return []

  rows = []

  # for each review add to rows with course and rating
  for review in data["reviews"]:
    rows.append({
      "professor": name,
      "review": review.get("review", ""),
      "stars": review.get("rating", None),
      "course": review.get("course", None)
    })

  return rows

In [25]:
reviews = []

# for each prof in my list, get reviews n add to reviews
for prof in profs:
  profreviews = getReviews(prof)
  reviews.extend(profreviews)

# create a dataframe w all the reviews
df = pd.DataFrame(reviews)

# parse data by removin any missing text or stars
df = df.dropna(subset=["review", "stars"])
df = df[df["review"].str.strip() != ""]
df["stars"] = df["stars"].astype(int)

In [26]:
# create my transformer model using hugging face model
## again in ref to piazza post @195
model = pipeline(
  "text-classification",
  model="nlptown/bert-base-multilingual-uncased-sentiment"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [27]:
# predicts the stars by taking text
def predictStars(text):
  # shorten reviews so it doesnt crash on me
  result = model(text[:512])[0]
  label = result["label"]
  return int(label[0])

In [28]:
# new dataframe rows for predicted, correct, and the error
df["predicted"] = df["review"].apply(predictStars)
df["correct"] = df["stars"] == df["predicted"]
df["error"] = abs(df["stars"] - df["predicted"])

In [30]:
# i wanted to print a summary df for each profs stats
summary = df.groupby("professor").agg(
  reviews = ("review", "count"),
  actual = ("stars", "mean"),
  predicted = ("predicted", "mean"),
  accuracy = ("correct", "mean"),
  error = ("error", "mean")
).reset_index()

print(summary)

            professor  reviews    actual  predicted  accuracy     error
0       Clyde Kruskal      111  2.612613   2.630631  0.522523  0.612613
1        Larry Herman      354  3.158192   3.073446  0.483051  0.672316
2     Maksym Morawski      269  2.855019   2.944238  0.423792  0.802974
3  Nelson Padua-Perez      250  4.236000   3.864000  0.552000  0.540000
4     Nora Burkhauser       58  2.310345   2.362069  0.431034  0.741379


In [31]:
# for accuracy and the avg error i used sklearn imports
accuracy = accuracy_score(df["stars"], df["predicted"])
mae = mean_absolute_error(df["stars"], df["predicted"])

# this is js my average scores between all profs
print("Accuracy:", accuracy)
print("Mean absolute error:", mae)

Accuracy: 0.4856046065259117
Mean absolute error: 0.6717850287907869
